In [2]:
import pandas as pd
import numpy as np

In [3]:
## test to make sure this file is up and running
print("hello world")

hello world


### Set up connection to MySQL database

In [4]:
## set up connection to MySQL database
import mysql.connector
from mysql.connector import Error


In [5]:
config = {
    'host': '172.29.218.123',
    'user': 'tg800',
    'password': 'b4R0Gm5psOvtmkPguM5aZFsF',
    'database': 'CGE'
}

### WTP import

In [15]:
WTP_RRI_csv_file = '/Users/thomasgooding/Desktop/WTP_RRI_TG.csv'
table_name = 'WTP_RRI_TG'
schema = 'WTP'

In [16]:
WTP_RRI_TG = pd.read_csv(WTP_RRI_csv_file, encoding='utf-8-sig')

WTP_RRI_TG.head()

,file,UniqID,Gender,Session,Study,Task,RRI,Unnamed: 7
0,0011FY13378815W6P.sta,11,F,1,W2,6P,151711.5,NaN
1,0011FY13378815WB1.sta,11,F,1,W2,B1,2299.6,NaN
2,0012FY15478010W6P.sta,12,F,1,W2,6P,39434.2,NaN
3,0012FY15478010WB1.sta,12,F,1,W2,B1,626.7,NaN
4,0012FY25478007W6P.sta,12,F,2,W2,6P,17010.0,NaN


In [17]:
conn = mysql.connector.connect(**config)

# **config unpacks the dictionary above into the connection settings
# Like logging into MySQL Workbench but through Python

cursor = conn.cursor()
## acts like a "pen" that exdecuted the SQL commands on the server

In [18]:
### Create a table in MySQL based on the columns in the dataframe and their data types
wtp_rri_cols = []
for col in WTP_RRI_df.columns:
    sample = WTP_RRI_df[col].dropna()  # looks at actual data in each column, ignoring blanks
    
    try:
        sample.astype(int)     # tries to convert column to integer
        col_type = "INT"
    except (ValueError, TypeError):
        try:
            sample.astype(float)   # tries to convert column to decimal number
            col_type = "FLOAT"
        except (ValueError, TypeError):
            max_len = sample.astype(str).str.len().max()  # checks longest value in column
            if max_len <= 50:
                col_type = "VARCHAR(50)"    # short text
            elif max_len <= 255:
                col_type = "VARCHAR(255)"   # medium text
            else:
                col_type = "TEXT"           # long text, no row size limit issues

    wtp_rri_cols.append(f"`{col}` {col_type}")

create_statement = f"CREATE TABLE IF NOT EXISTS `{schema}`.`{table_name}` ({', '.join(wtp_rri_cols)});"
cursor.execute(create_statement)
print(f"Table `{table_name}` is ready!")

Table `WTP_RRI_TG` is ready!


In [19]:
## Insert data

inserted = 0

for _, row in WTP_RRI_TG.iterrows():
    # df.itterows() loops through every row in the csv file one at a time
    values = [None if pd.isna(v) else str(v) for v in row]
    ## converts each value to a string and returns empty/blank cells into Nulls for MySQL
    placeholders = ','.join(['%s'] * len(values))

    col_names = ', '.join([f'`{c}`' for c in WTP_RRI_TG.columns])
    # builds the column name list for the INSERT statement
    
    insert_statement = f"INSERT INTO `{schema}`.`{table_name}` ({col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)  # executes the INSERT with actual values
    inserted += 1

    # ==============================
# STEP 8: Save changes and close connection
# ==============================
conn.commit()  
## .commit() is like hitting "save" - without this your inserts won't actually save

print(f"Done! {inserted} rows successfully inserted.")

Done! 310 rows successfully inserted.


### Run this query in MySQL to insert RRI data into existing WTP.HRV table

-- SELECT * FROM WTP.HRV;

-- SELECT * FROM WTP.WTP_RRI_TG;

SET SQL_SAFE_UPDATES = 0;

UPDATE WTP.HRV hrv
JOIN WTP.WTP_RRI_TG tg
  ON tg.UniqID = hrv.sub_id
  AND tg.Task = hrv.sub_task
  AND tg.Session = hrv.session
SET hrv.RRI = tg.RRI;

SET SQL_SAFE_UPDATES = 1;


##$ run the same for Vt1 RRI to get into VT1.HRV

-- SELECT * FROM WTP.HRV;

-- SELECT * FROM WTP.WTP_RRI_TG;

SET SQL_SAFE_UPDATES = 0;

UPDATE WTP.HRV hrv
JOIN WTP.WTP_RRI_TG tg
  ON tg.UniqID = hrv.sub_id
  AND tg.Task = hrv.sub_task
  AND tg.Session = hrv.session
SET hrv.RRI = tg.RRI;

SET SQL_SAFE_UPDATES = 1;

In [46]:
## lets upload a csv file to test how well this works
## WTP daemographic data
WTP_csv_file = '/Users/thomasgooding/Desktop/SQL data to import/WTP_dem_import.csv'
table_name = 'WTP_dem_TG'
schema = 'WTP'

In [14]:
## need to strip the BOM character which was preventig me from using import wizard in MySQL workbench. 
## I can modify the original csv file to a format without the BOM character, but the import wizard was slow...testing this method to see if it's ultimately faster for the several csv files I need to import.

wtp_df = pd.read_csv(WTP_csv_file, encoding='utf-8-sig')  

wtp_df.columns = wtp_df.columns.str.strip()

## get confirmation that the csv file was loaded correctly and that the rows/columns are as expected.
print(f"CSV loaded successfullly! Found {len(wtp_df)} rows and {len(wtp_df.columns)} columns")
print(f"Columns: {list(wtp_df.columns)}")

CSV loaded successfullly! Found 175 rows and 832 columns
Columns: ['IDNUM', 'COHORT', 'CSDATE', 'BLDATE', 'IN1', 'IN2', 'IN3', 'HEIGHTF', 'HEIGHTI', 'HEIGHT', 'IN4', 'IN4a', 'IN5', 'IN6', 'IN7', 'IN8', 'IN9', 'IN9A1', 'IN9H1', 'IN9A2', 'IN9H2', 'IN9A3', 'IN9H3', 'IN9A4', 'IN9H4', 'IN9A5', 'IN9H5', 'IN10', 'IN10A1', 'IN10H1', 'IN10A2', 'IN10H2', 'IN10A3', 'IN10H3', 'IN10A4', 'IN10H4', 'IN10A5', 'IN10H5', 'IN11', 'IN12', 'IN13', 'IN14', 'IN15', 'IN16', 'IN17', 'IN18', 'IN28', 'IN28_2', 'IN28_3', 'IN28_4', 'IN28_5', 'IN29', 'IN29_2', 'IN29_3', 'IN29_4', 'IN29_5', 'IN30', 'IN30_2', 'IN30_3', 'IN30_4', 'IN30_5', 'IN31', 'IN31_2', 'IN31_3', 'IN31_4', 'IN31_5', 'IN32', 'IN32_2', 'IN32_3', 'IN32_4', 'IN32_5', 'IN33', 'IN33_2', 'IN33_3', 'IN33_4', 'IN33_5', 'IN34', 'IN34_2', 'IN34_3', 'IN34_4', 'IN34_5', 'IN35', 'IN35_2', 'IN35_3', 'IN35_4', 'IN35_5', 'IN36', 'IN36_2', 'IN36_3', 'IN36_4', 'IN36_5', 'IN37', 'IN37_2', 'IN37_3', 'IN37_4', 'IN37_5', 'IN38', 'IN38_2', 'IN38_3', 'IN38_4', 'IN38_5', '

In [16]:
conn = mysql.connector.connect(**config)

# **config unpacks the dictionary above into the connection settings
# Like logging into MySQL Workbench but through Python

cursor = conn.cursor()
## acts like a "pen" that exdecuted the SQL commands on the server

In [28]:
### Create a table in MySQL based on the columns in the dataframe and their data types

wtp_cols = []
for col in wtp_df.columns:
    sample = wtp_df[col].dropna()  # looks at actual data in each column, ignoring blanks
    
    try:
        sample.astype(int)     # tries to convert column to integer
        col_type = "INT"
    except (ValueError, TypeError):
        try:
            sample.astype(float)   # tries to convert column to decimal number
            col_type = "FLOAT"
        except (ValueError, TypeError):
            max_len = sample.astype(str).str.len().max()  # checks longest value in column
            if max_len <= 50:
                col_type = "VARCHAR(50)"    # short text
            elif max_len <= 255:
                col_type = "VARCHAR(255)"   # medium text
            else:
                col_type = "TEXT"           # long text, no row size limit issues

    wtp_cols.append(f"`{col}` {col_type}")

create_statement = f"CREATE TABLE IF NOT EXISTS `{schema}`.`{table_name}` ({', '.join(wtp_cols)});"
cursor.execute(create_statement)
print(f"Table `{table_name}` is ready!")

Table `WTP_dem_TG` is ready!


In [29]:
## Insert data

inserted = 0

for _, row in wtp_df.iterrows():
    # df.itterows() loops through every row in the csv file one at a time
    values = [None if pd.isna(v) else str(v) for v in row]
    ## converts each value to a string and returns empty/blank cells into Nulls for MySQL
    placeholders = ','.join(['%s'] * len(values))

    col_names = ', '.join([f'`{c}`' for c in wtp_df.columns])
    # builds the column name list for the INSERT statement
    
    insert_statement = f"INSERT INTO `{schema}`.`{table_name}` ({col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)  # executes the INSERT with actual values
    inserted += 1

    # ==============================
# STEP 8: Save changes and close connection
# ==============================
conn.commit()  
## .commit() is like hitting "save" - without this your inserts won't actually save

print(f"Done! {inserted} rows successfully inserted.")

Done! 175 rows successfully inserted.


## K24 data to import



In [12]:
k24_dem_df = pd.read_csv(
    '/Users/thomasgooding/Desktop/SQL data to import/k24_dem_tg.csv', 
    encoding='latin1')

k24_dem_df.columns = k24_dem_df.columns.str.strip()
k24_table_name = 'k24_dem_TG'
k24_schema = 'K24'

print(f"CSV loaded: {len(k24_dem_df)} rows, {len(k24_dem_df.columns)} columns")


CSV loaded: 139 rows, 25 columns


In [13]:
conn = mysql.connector.connect(**config)
cursor = conn.cursor()
print("Connected to MySQL!")


Connected to MySQL!


In [14]:
k24_cols = []
for col in k24_dem_df.columns:
    sample = k24_dem_df[col].dropna()
    
    try:
        sample.astype(int)
        col_type = "INT"
    except (ValueError, TypeError):
        try:
            sample.astype(float)
            col_type = "FLOAT"
        except (ValueError, TypeError):
            max_len = sample.astype(str).str.len().max()
            if max_len <= 50:
                col_type = "VARCHAR(50)"
            elif max_len <= 255:
                col_type = "VARCHAR(255)"
            else:
                col_type = "TEXT"

    k24_cols.append(f"`{col}` {col_type}")

In [15]:
# DROP OLD TABLE AND RECREATE WITH utf8mb4
# cursor.execute(f"DROP TABLE IF EXISTS `{k24_schema}`.`{k24_table_name}`;")
# print(f"Old table dropped (if it existed)")

create_statement = f"""
    CREATE TABLE `{k24_schema}`.`{k24_table_name}` 
    ({', '.join(k24_cols)}) 
    CHARACTER SET utf8mb4 
    COLLATE utf8mb4_unicode_ci;
"""
cursor.execute(create_statement)
print(f"Table `{k24_table_name}` created with utf8mb4 charset!")

Table `k24_dem_TG` created with utf8mb4 charset!


In [16]:
## double check the column definitions
k24_cols

['`ID6` VARCHAR(50)',
 '`DATE` VARCHAR(50)',
 '`SESS` VARCHAR(50)',
 '`RA` VARCHAR(50)',
 '`TIME` VARCHAR(50)',
 '`DRUG24` VARCHAR(255)',
 '`ALC24` VARCHAR(255)',
 '`BAC` VARCHAR(50)',
 '`RegBP_1` VARCHAR(50)',
 '`RegBP_2` VARCHAR(50)',
 '`BP1_1` VARCHAR(50)',
 '`BP1_2` VARCHAR(50)',
 '`FHxBP` VARCHAR(255)',
 '`FHxBP_2_TEXT` VARCHAR(255)',
 '`TEMP` VARCHAR(50)',
 '`MENS` VARCHAR(50)',
 '`HEIGHT` VARCHAR(50)',
 '`WEIGHT` VARCHAR(50)',
 '`ARM` VARCHAR(50)',
 '`PREG` VARCHAR(50)',
 '`BP2_1` VARCHAR(50)',
 '`BP2_2` VARCHAR(50)',
 '`CARD` VARCHAR(50)',
 '`NOTES` VARCHAR(255)',
 '`EXCLUDED` VARCHAR(255)']

In [19]:
inserted = 0
for _, row in k24_dem_df.iterrows():
    values        = [None if pd.isna(v) else str(v) for v in row]
    placeholders  = ', '.join(['%s'] * len(values))
    k24_col_names = ', '.join([f'`{c}`' for c in k24_dem_df.columns])
    
    insert_statement = f"INSERT INTO `{k24_schema}`.`{k24_table_name}` ({k24_col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)
    inserted += 1

conn.commit()
print(f"Done! {inserted} rows inserted into `{k24_schema}`.`{k24_table_name}`")

Done! 139 rows inserted into `K24`.`k24_dem_TG`


### Stress_pic data import

In [22]:
stress_pic_dem_df = pd.read_csv(
    '/Users/thomasgooding/Desktop/SQL data to import/stress_pic_dem_tg.csv',
    encoding='latin1')

stress_pic_dem_df.columns = stress_pic_dem_df.columns.str.strip()
stress_pic_table_name = 'stress_pic_dem_TG'
stress_pic_schema = 'stress_pic'

print(f"CSV loaded: {len(stress_pic_dem_df)} rows, {len(stress_pic_dem_df.columns)} columns")

CSV loaded: 242 rows, 76 columns


In [23]:
conn = mysql.connector.connect(**config)

cursor = conn.cursor()

In [24]:
stress_pic_cols= []

for col in stress_pic_dem_df.columns:
    sample = stress_pic_dem_df[col].dropna()  # looks at actual data in each column, ignoring blanks
    
    try:
        sample.astype(int)     # tries to convert column to integer
        col_type = "INT"
    except (ValueError, TypeError):
        try:
            sample.astype(float)   # tries to convert column to decimal number
            col_type = "FLOAT"
        except (ValueError, TypeError):
            max_len = sample.astype(str).str.len().max()  # checks longest value in column
            if max_len <= 50:
                col_type = "VARCHAR(50)"    # short text
            elif max_len <= 255:
                col_type = "VARCHAR(255)"   # medium text
            else:
                col_type = "TEXT"           # long text, no row size limit issues

    stress_pic_cols.append(f"`{col}` {col_type}")

In [25]:
stress_pic_cols

['`id6` INT',
 '`gender` VARCHAR(50)',
 '`female` INT',
 '`history` INT',
 '`CONDITION1` VARCHAR(50)',
 '`CONDITION2` VARCHAR(50)',
 '`CONDITION1x` INT',
 '`CONDITION2x` INT',
 '`SBP1` INT',
 '`DBP1` INT',
 '`SBP2` INT',
 '`DBP2` INT',
 '`WEIGHT` INT',
 '`HEIGHT` INT',
 '`ARM` INT',
 '`PERIOD` INT',
 '`PREDR` INT',
 '`POSTDR` INT',
 '`POSTDR5` INT',
 '`POSTDR10` INT',
 '`POSTDR15` INT',
 '`POSTDR20` INT',
 '`POSTDR25` INT',
 '`POSTDR30` INT',
 '`POSTDR35` INT',
 '`POSTDR40` INT',
 '`POSTDR45` INT',
 '`POSTDR50` INT',
 '`POSTDR55` INT',
 '`POSTDR60` INT',
 '`MEMPRE1` INT',
 '`MEMPRE2` INT',
 '`MEMPOST3` INT',
 '`POST30` INT',
 '`POST60` INT',
 '`POST90` INT',
 '`POST120` INT',
 '`POST150` INT',
 '`POST180` INT',
 '`POST210` INT',
 '`POST240` INT',
 '`POST270` INT',
 '`POST300` INT',
 '`POST330` INT',
 '`POST360` INT',
 '`TimeBD` INT',
 '`TimeDR` INT',
 '`TimeR5` INT',
 '`TimeR10` INT',
 '`TimeR15` INT',
 '`TimeR20` INT',
 '`TimeR25` INT',
 '`TimeR30` INT',
 '`TimeR35` INT',
 '`TimeR40` 

In [26]:
create_statement = f"CREATE TABLE IF NOT EXISTS `{stress_pic_schema}`.`{stress_pic_table_name}` ({', '.join(stress_pic_cols)});"
cursor.execute(create_statement)
print(f"Table `{stress_pic_table_name}` is ready!")


Table `stress_pic_dem_TG` is ready!


In [31]:
inserted = 0
for _, row in stress_pic_dem_df.iterrows():
    values        = [None if pd.isna(v) else str(v) for v in row]
    placeholders  = ', '.join(['%s'] * len(values))
    stress_pic_col_names = ', '.join([f'`{c}`' for c in stress_pic_dem_df.columns])

    insert_statement = f"INSERT INTO `{stress_pic_schema}`.`{stress_pic_table_name}` ({stress_pic_col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)
    inserted += 1

conn.commit()
print(f"Done! {inserted} rows inserted into `{stress_pic_schema}`.`{stress_pic_table_name}`")

Done! 242 rows inserted into `stress_pic`.`stress_pic_dem_TG`


### Next is future steps for improving this python code to import all needed csv files into MYSQL workbench. 



In [ ]:
# # ==============================
# # STEP 8: Define all your CSV files, their table names, and schemas
# # ==============================
# files_to_import = [
#     {
#         'csv_file':   '/Users/thomasgooding/Documents/WTP_dem_import.csv',
#         'table_name': 'WTP_dem_TG',
#         'schema':     'WTP'
#     },
#     {
#         'csv_file':   '/Users/thomasgooding/Documents/CGE_dem_TG.csv',
#         'table_name': 'CGE_dem_TG',
#         'schema':     'CGE'
#     },
#     {
#         'csv_file':   '/Users/thomasgooding/Documents/VT1_dem_TG.csv',
#         'table_name': 'VT1_dem_TG',
#         'schema':     'VT1'
#     },
#     # add as many files as you need following the same format
# ]

# # ==============================
# # STEP 9: Loop through each file and import it
# # ==============================
# for file in files_to_import:
    
#     csv_file   = file['csv_file']
#     table_name = file['table_name']
#     schema     = file['schema']        # pulls the schema for each individual file
    
#     # Read the CSV
#     df = pd.read_csv(csv_file, encoding='utf-8-sig')
#     df.columns = df.columns.str.strip()
#     print(f"\nImporting {table_name} into schema {schema}... ({len(df)} rows, {len(df.columns)} columns)")

#     # Create table with smart data types
#     cols = []
#     for col in df.columns:
#         sample = df[col].dropna()
#         try:
#             sample.astype(int)
#             col_type = "INT"
#         except (ValueError, TypeError):
#             try:
#                 sample.astype(float)
#                 col_type = "FLOAT"
#             except (ValueError, TypeError):
#                 max_len = sample.astype(str).str.len().max()
#                 if max_len <= 50:
#                     col_type = "VARCHAR(50)"
#                 elif max_len <= 255:
#                     col_type = "VARCHAR(255)"
#                 else:
#                     col_type = "TEXT"
#         cols.append(f"`{col}` {col_type}")

#     create_statement = f"CREATE TABLE IF NOT EXISTS `{schema}`.`{table_name}` ({', '.join(cols)});"
#     cursor.execute(create_statement)

#     # Insert rows
#     inserted = 0
#     for _, row in df.iterrows():
#         values           = [None if pd.isna(v) else str(v) for v in row]
#         placeholders     = ', '.join(['%s'] * len(values))
#         col_names        = ', '.join([f'`{c}`' for c in df.columns])
#         insert_statement = f"INSERT INTO `{schema}`.`{table_name}` ({col_names}) VALUES ({placeholders});"
#         cursor.execute(insert_statement, values)
#         inserted += 1

#     conn.commit()
#     print(f"Done! {inserted} rows inserted into `{schema}`.`{table_name}`")

# # ==============================
# # STEP 10: Close connection after ALL files are done
# # ==============================
# cursor.close()
# conn.close()
# print("\nAll files imported successfully! Connection closed.")